# ArcGIS Pro Preprocessing Workflow
## El Rio Lobo (ERL) 2023 Field Trial

### Companion workflow for

**Detka, J. R., Purdy, A. J., Melton, F. S., Daugovish, O., Greer, C. A., & Martin, F. N. (2026).**
*A Plant-Level Survival Modeling Framework for Spatiotemporal Strawberry Canopy Decline Using UAV Multispectral Time Series.*
**Drones, 10**(4), 235.

https://doi.org/10.3390/drones10040235

---

### Purpose

This notebook documents the ArcGIS Pro / ArcPy preprocessing workflow used to transform UAV multispectral imagery into plant-level spatial datasets for downstream survival analysis and machine learning.

The workflow includes:

- Clip multispectral orthomosaics by treatment block
- Calculate vegetation indices (NDVI, NDRE, and Redness Index)
- Delineate individual strawberry plants
- Generate plant centroids
- Create standardized plant hexagons
- Classify canopy condition
- Calculate plant-level canopy metrics
- Export datasets for downstream Python and R analyses

### Provenance

This notebook is the original ArcGIS Pro workflow used during development of the methods presented in the accompanying publication. It has been lightly edited for readability and documentation, but the analytical workflow and processing steps remain unchanged.

> **Note:** Large UAV imagery, Pix4D products, intermediate rasters, and other project datasets are not included in this repository because of their size. Users should configure local paths to their own data before running the workflow.


## Step 1 - Clip multispectral rasters by treatment block

Clips large Pix4D multispectral orthomosaics to each field/treatment block.


In [ ]:
from config import *

import arcpy
import os

# Set up workspace and paths
workspace = PROJECT_ROOT.as_posix()

# These folders now come from config.py
output_folder = CLIPPED_RASTERS.as_posix()
treatment_blocks_folder = TREATMENT_BLOCKS.as_posix()

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Folder containing the treatment blocks
treatment_blocks_folder = os.path.join(workspace, "Vectors", "Treatment_Blocks")

# List of raster files
raster_files = [
    "Pix4D_USDA_2023_06_20_multispectral_composite.tif",
    "Pix4D_USDA_2023_06_10_multispectral_composite.tif",
    "Pix4D_USDA_2023_06_01_multispectral_composite.tif",
    "Pix4D_USDA_2023_05_22_multispectral_composite.tif",
    "Pix4D_USDA_2023_05_15_multispectral_composite.tif",
    "Pix4D_USDA_2023_04_21_multispectral_composite.tif",
    "Pix4D_USDA_2023_03_28_multispectral_composite.tif",
    "Pix4D_USDA_2023_02_20_multispectral_composite.tif",
    "Pix4D_USDA_2023_02_09_multispectral_composite.tif",
    "Pix4D_USDA_2022_12_07_multispectral_composite.tif"
]

# Loop through each raster and clip by each treatment block
for raster_file in raster_files:
    raster_path = os.path.join(RASTERS.as_posix(), raster_file)
    
    # Extract the date part from the raster file name (YYYY_MM_DD format)
    date = '_'.join(raster_file.split('_')[2:5])  # e.g., "2023_06_20"

    print(f"🗓️ Processing date: {date}")

    # Loop through each treatment block (Field_1 to Field_4)
    for field_num in FIELDS:
        # Define the treatment block shapefile path
        treatment_block_shapefile = os.path.join(treatment_blocks_folder, f"Field_{field_num}.shp")

        # Define output file name using the desired format
        output_raster = os.path.join(output_folder, f"ERL_Field{field_num}_{date}_multispectral.tif")
        
        # Check if the clipped raster already exists
        if os.path.exists(output_raster):
            print(f"⏩ Skipping Field {field_num} on {date}, file already exists: {output_raster}")
            continue  # Skip this iteration if the file exists

        # Perform the clipping
        arcpy.management.Clip(raster_path, "", output_raster, treatment_block_shapefile, nodata_value="NONE", clipping_geometry="ClippingGeometry")

        # Print progress with fun emojis
        print(f"🚀 Clipped raster for Field {field_num} on {date} is done! 🎉 Saved as {output_raster}")



## Step 2 - Calculate NDVI, plant masks, and initial plant polygons

Calculates NDVI from red and NIR bands, thresholds plant material, polygonizes the binary plant mask, and adds layers to the ArcGIS Pro map.


In [ ]:
print("✅ All rasters have been successfully processed! 🎯")

import arcpy
import os
from arcpy.sa import *

# Check out the Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Set up workspace and paths
workspace = CLIPPED_RASTERS.as_posix()

output_folder = NDVI_OUTPUT.as_posix()
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# List of clipped rasters to process (e.g., for the date 2022_12_07)
raster_files = [
    "ERL_Field1_2022_12_07_multispectral.tif",
    "ERL_Field2_2022_12_07_multispectral.tif",
    "ERL_Field3_2022_12_07_multispectral.tif",
    "ERL_Field4_2022_12_07_multispectral.tif"
]
# NDVI threshold
ndvi_threshold = 0.20  # Updated threshold to focus on strawberry plants

# Loop through each clipped field raster
for raster_file in raster_files:
    raster_path = os.path.join(workspace, raster_file)
    
    # Extract field number and date
    field_name = raster_file.split('_')[1]  # Extracts "Field1", "Field2", etc.
    date = '_'.join(raster_file.split('_')[2:5])  # Extracts "2022_12_07"

    print(f"🚀 Processing {field_name} for date {date}")

    # Define output paths for NDVI, binary plant mask, and polygons
    ndvi_output_path = os.path.join(output_folder, f"ERL_NDVI_{field_name}_{date}.tif")
    binary_plant_mask_output_path = os.path.join(output_folder, f"ERL_binary_plant_mask_{field_name}_{date}.tif")
    polygons_output_path = os.path.join(output_folder, f"ERL_polygons_{field_name}_{date}.shp")

    # Calculate NDVI using Band 1 (Red) and Band 4 (NIR)
    print(f"🧮 Calculating NDVI for {field_name}...")
    red_band = Raster(raster_path + r"/Band_1")
    nir_band = Raster(raster_path + r"/Band_4")

    # NDVI calculation
    ndvi = (nir_band - red_band) / (nir_band + red_band)
    ndvi.save(ndvi_output_path)  # Ensure NDVI is saved with correct naming
    print(f"✅ NDVI calculated for {field_name} and saved to {ndvi_output_path}")

    # Apply thresholding to identify plants based on adjusted NDVI threshold
    print(f"🔍 Identifying plants in {field_name}...")
    binary_plant_mask = Con(ndvi >= ndvi_threshold, 1, 0)
    binary_plant_mask.save(binary_plant_mask_output_path)
    print(f"✅ Binary plant mask created for {field_name} at {binary_plant_mask_output_path}")

    # Polygonize the binary plant mask
    print(f"🗺️ Polygonizing binary plant mask for {field_name}...")
    arcpy.conversion.RasterToPolygon(binary_plant_mask_output_path, polygons_output_path, "NO_SIMPLIFY", "VALUE")
    print(f"✅ Polygons created for {field_name} at {polygons_output_path}")

    # Open the current ArcGIS Pro project and the first map
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    mp = aprx.listMaps()[0]

    # Add NDVI layer to ArcGIS Pro contents with custom name
    print(f"🖼️ Checking if NDVI layer is already in contents for {field_name}...")
    ndvi_layer_exists = any(
        hasattr(layer, 'dataSource') and layer.dataSource == ndvi_output_path
        for layer in mp.listLayers()
    )
    
    if not ndvi_layer_exists:
        print(f"🖼️ Adding NDVI layer to ArcGIS Pro contents for {field_name}...")
        ndvi_layer = mp.addDataFromPath(ndvi_output_path)
        ndvi_layer.name = f"NDVI_{field_name}_{date}"
        print(f"✅ NDVI layer added to ArcGIS Pro contents for {field_name}.")
    else:
        print(f"✅ NDVI layer already exists for {field_name}.")

    # Add Polygon layer to ArcGIS Pro contents with no fill and custom name
    print(f"🖼️ Checking if Polygon layer is already in contents for {field_name}...")
    polygon_layer_exists = any(
        hasattr(layer, 'dataSource') and layer.dataSource == polygons_output_path
        for layer in mp.listLayers()
    )
    
    if not polygon_layer_exists:
        print(f"🖼️ Adding Polygon layer to ArcGIS Pro contents for {field_name} with no fill...")
        polygon_layer = mp.addDataFromPath(polygons_output_path)
        polygon_layer.name = f"Polygons_{field_name}_{date}"

        # Update symbology: No fill, visible outline
        sym = polygon_layer.symbology
        sym.updateRenderer('SimpleRenderer')
        sym.renderer.symbol.symbolType = "EsriSFS"  # Simple Fill Symbol
        sym.renderer.symbol.color = {'RGB': [0, 0, 0, 0]}  # No fill (transparent)
        sym.renderer.symbol.outlineColor = {'RGB': [255, 0, 0, 255]}  # Red outline
        sym.renderer.symbol.outlineWidth = 1.5  # Outline width
        polygon_layer.symbology = sym
        print(f"✅ Polygon layer added and styled for {field_name}.")
    else:
        print(f"✅ Polygon layer already exists for {field_name}.")

    # Add Binary Plant Mask layer to ArcGIS Pro contents with custom name
    print(f"🖼️ Adding Binary Plant Mask layer to ArcGIS Pro contents for {field_name}...")
    plant_mask_layer = mp.addDataFromPath(binary_plant_mask_output_path)
    plant_mask_layer.name = f"Binary_Plant_Mask_{field_name}_{date}"
    print(f"✅ Binary Plant Mask layer added for {field_name}.")



## Step 3 - Generate plant centroids by bed row

Creates centroid points from plant polygons intersecting each bed-row buffer and combines per-row centroids into a single shapefile.


In [ ]:
print("🎉 All fields processed, NDVI calculated, polygons polygonized, and displayed with no fill!")

import arcpy
import os

# Set workspace and paths
workspace = VECTORS.as_posix()
buffer_shapefile = os.path.join(workspace, r"ERL_Field2_2022_12_07_Row_Buffers.shp")  # Path to row buffer shapefile
plant_polygons = os.path.join(workspace, "ERL_polygons_Field2_2022_12_07.shp")  # Path to plant polygons shapefile
output_centroid_folder = CENTROID_OUTPUT.as_posix()
output_combined_shapefile = os.path.join(output_centroid_folder, "Combined_Centroids_Field2_2022_12_07.shp")

# Create necessary directories if they don't exist
if not os.path.exists(output_centroid_folder):
    os.makedirs(output_centroid_folder)

# Create layers for row buffer and plant polygons
bed_buffer_layer = arcpy.management.MakeFeatureLayer(buffer_shapefile, "bed_buffer_layer")
plant_polygon_layer = arcpy.management.MakeFeatureLayer(plant_polygons, "plant_polygon_layer")

# Open the current ArcGIS Pro project and get the first map
aprx = arcpy.mp.ArcGISProject("CURRENT")
mp = aprx.listMaps()[0]

# Loop through each unique bed (identified by ORIG_FID) in the buffer shapefile
with arcpy.da.SearchCursor(buffer_shapefile, ["ORIG_FID"]) as cursor:
    for row in cursor:
        orig_fid = row[0]
        print(f"🚀 Processing bed row with ORIG_FID: {orig_fid}...")

        # Select the specific bed row buffer by ORIG_FID
        arcpy.management.SelectLayerByAttribute(bed_buffer_layer, "NEW_SELECTION", f"ORIG_FID = {orig_fid}")

        # Select plant polygons that intersect with the selected bed buffer
        arcpy.management.SelectLayerByLocation(plant_polygon_layer, "INTERSECT", bed_buffer_layer)

        # Generate centroid points for the selected plant polygons
        centroid_output_path = os.path.join(output_centroid_folder, f"Centroids_Bed{orig_fid}_Field2_2022_12_07.shp")
        arcpy.management.FeatureToPoint(plant_polygon_layer, centroid_output_path, "INSIDE")

        print(f"✅ Centroid points generated and saved to {centroid_output_path} for bed row {orig_fid}.")

        # Check if 'Plant_Id' exists before adding
        field_names = [f.name for f in arcpy.ListFields(centroid_output_path)]
        if "Plant_Id" not in field_names:
            arcpy.management.AddField(centroid_output_path, "Plant_Id", "LONG")
            print(f"✅ Added 'Plant_Id' field to {centroid_output_path}")
        
        # Populate 'Plant_Id' field only if it was just added or needs updating
        if "Plant_Id" in field_names:
            with arcpy.da.UpdateCursor(centroid_output_path, ["Id", "Plant_Id"]) as update_cursor:
                for update_row in update_cursor:
                    if update_row[1] is None:  # Only update if the value is missing
                        update_row[1] = update_row[0]  # Copy 'Id' to 'Plant_Id'
                        update_cursor.updateRow(update_row)
            print(f"🔢 Populated 'Plant_Id' field for {centroid_output_path}")
            print(f"🔢 Plant_Id field populated for centroids in bed row {orig_fid}.")

        # Get existing fields
        field_names = [f.name for f in arcpy.ListFields(centroid_output_path)]
        
        # Add 'Bed_Row' field only if it does not exist
        if "Bed_Row" not in field_names:
            arcpy.management.AddField(centroid_output_path, "Bed_Row", "TEXT")
            print(f"✅ Added 'Bed_Row' field to {centroid_output_path}")
        
        # Populate 'Bed_Row' field only if it was just added or needs updating
        with arcpy.da.UpdateCursor(centroid_output_path, ["Bed_Row"]) as bed_cursor:
            for bed_row in bed_cursor:
                if bed_row[0] is None or bed_row[0] == "":  # Only update if missing
                    bed_row[0] = f"{orig_fid}"  # Set Bed_Row value
                    bed_cursor.updateRow(bed_row)
        print(f"🔢 Populated 'Bed_Row' field for {centroid_output_path}")
        print(f"🔢 Bed_Row field populated for centroids in bed row {orig_fid}.")

        # Add the centroid layer to ArcGIS Pro for visualization (if needed)
        mp.addDataFromPath(centroid_output_path)

# List all centroid shapefiles that were generated
centroid_files = [f for f in os.listdir(output_centroid_folder) if f.startswith("Centroids_Bed") and f.endswith(".shp")]

# Print the files found in the directory
print(f"🗂️ Found {len(centroid_files)} centroid shapefiles:")
for f in centroid_files:
    print(f" - {f}")

# Combine the centroids into a single shapefile
if len(centroid_files) == 0:
    print("❌ No centroid shapefiles found. Please check the directory and file names.")
else:
    # Check if the combined shapefile already exists; if so, delete it
    if arcpy.Exists(output_combined_shapefile):
        arcpy.management.Delete(output_combined_shapefile)

    # Loop through each centroid file and append them to the combined shapefile
    for centroid_file in centroid_files:
        centroid_path = os.path.join(output_centroid_folder, centroid_file)
        
        # If this is the first file, create the combined shapefile
        if not arcpy.Exists(output_combined_shapefile):
            # Copy the first centroid shapefile to start the combined shapefile
            arcpy.management.CopyFeatures(centroid_path, output_combined_shapefile)
        
        # If the combined shapefile already exists, append new centroids
        else:
            arcpy.management.Append(centroid_path, output_combined_shapefile, "NO_TEST")

        print(f"✅ Centroids from {centroid_file} added to combined shapefile.")

print(f"🎉 All centroid shapefiles combined into {output_combined_shapefile}.")


## Step 4 - Average near-duplicate centroids

Finds centroids within a distance threshold and replaces close pairs with averaged points.


In [ ]:

import arcpy
import math
import os

workspace = VECTORS.as_posix()
combined_centroids = os.path.join(
    CENTROID_OUTPUT.as_posix(),
    "Combined_Centroids_Field2_2022_12_07.shp"
)

# Set the distance threshold (e.g., 0.16 meters)
distance_threshold = 0.16

# Open the combined centroid shapefile
bed_buffer_layer = arcpy.management.MakeFeatureLayer(combined_centroids, "bed_buffer_layer")

# Iterate over each bed row based on 'Bed_Row' attribute
with arcpy.da.SearchCursor(combined_centroids, ["Bed_Row"]) as bed_cursor:
    unique_beds = set(row[0] for row in bed_cursor)

# Process centroids for each unique bed row
for bed_row in unique_beds:
    print(f"🚀 Processing centroids for bed row: {bed_row}...")

    # Select centroids that belong to the current bed row
    arcpy.management.SelectLayerByAttribute(bed_buffer_layer, "NEW_SELECTION", f"Bed_Row = '{bed_row}'")

    # Use an update cursor to process the centroids
    with arcpy.da.UpdateCursor(bed_buffer_layer, ["SHAPE@XY", "Plant_Id", "OID@"]) as cursor:
        points_to_remove = []  # Store OIDs of points to be removed
        points_to_add = []  # Store new points (averaged) to add

        # Convert cursor to list for easier iteration
        point_list = list(cursor)

        # Iterate through each point in the selected bed row
        for i, (point1, plant_id1, oid1) in enumerate(point_list):
            for j, (point2, plant_id2, oid2) in enumerate(point_list):
                if i >= j:
                    continue

                # Calculate distance between points
                distance = math.sqrt((point2[0] - point1[0])**2 + (point2[1] - point1[1])**2)

                # If distance is less than threshold, average points
                if distance < distance_threshold:
                    avg_x = (point1[0] + point2[0]) / 2
                    avg_y = (point1[1] + point2[1]) / 2
                    new_point = ((avg_x, avg_y), plant_id1)  # Keep plant_id1

                    # Mark the original points for removal
                    points_to_remove.extend([oid1, oid2])

                    # Add new averaged point for insertion
                    points_to_add.append(new_point)

        # Delete the original points that were marked for removal
        cursor.reset()
        for row in cursor:
            if row[2] in points_to_remove:
                cursor.deleteRow()

    # Insert the new averaged points
    with arcpy.da.InsertCursor(combined_centroids, ["SHAPE@XY", "Plant_Id"]) as insert_cursor:
        for new_point in points_to_add:
            insert_cursor.insertRow(new_point)

    print(f"✅ Processed and updated centroids for bed row: {bed_row}")


## Step 5 - Check duplicate plant IDs

Counts repeated plant identifiers as a quality-control check.


In [ ]:

print("🎉 All bed rows processed and updated centroids added.")

import arcpy
from collections import Counter

# Set the path to the combined centroids shapefile
combined_centroids = os.path.join(
    CENTROID_OUTPUT.as_posix(),
    "Combined_Centroids_Field2_2022_12_07.shp"
)


# Create a list to store all Plant_Id values
plant_ids = []

# Use a SearchCursor to loop through the shapefile and collect Plant_Id values
with arcpy.da.SearchCursor(combined_centroids, ["Id"]) as cursor:
    for row in cursor:  # Ensure indentation is consistent (use spaces, not tabs)
        plant_ids.append(row[0])

# Use Counter from the collections module to count occurrences of each Plant_Id
plant_id_counts = Counter(plant_ids)

# Find duplicates (Plant_Id values with more than 1 occurrence)
duplicates = {plant_id: count for plant_id, count in plant_id_counts.items() if count > 1}

# Print results
if duplicates:
    print(f"🔍 Found {len(duplicates)} duplicate Plant_Id values:")
    for plant_id, count in duplicates.items():
        print(f" - Plant_Id: {plant_id}, Count: {count}")
else:
    print("✅ No duplicate Plant_Id values found.")




## Step 6 - Create plant hexagons from centroids

Creates fixed-size hexagonal plant sampling units and assigns Plant_Id and Bed_Row attributes.


In [ ]:


import arcpy
from math import cos, sin, pi
import os

# Set workspace and file paths
workspace = VECTORS.as_posix()

polygon_file = os.path.join(workspace, "ERL_polygons_clipped_Field2_2022_12_07.shp")

centroid_file = os.path.join(
    CENTROID_OUTPUT.as_posix(),
    "Combined_Centroids_Field2_2022_12_07.shp"
)

row_buffer_file = os.path.join(workspace, "ERL_Field2_2022_12_07_Row_Buffers.shp")
hexagon_output = os.path.join(workspace, "ERL_Field2_2022_12_07_PlantHex.shp")
hex_size = 0.17  # Size of hexagons (buffer radius)

# Function to create a hexagon
def create_hexagon(center, size):
    hex_coords = []
    for i in range(6):
        angle = pi / 3 * i
        x = center[0] + size * cos(angle)
        y = center[1] + size * sin(angle)
        hex_coords.append([x, y])
    hex_coords.append(hex_coords[0])  # Close the hexagon
    return hex_coords

# Step 1: Copy Id to Plant_Id in the polygon file (if needed)
def copy_id_to_plant_id(polygon_file):
    fields = [f.name for f in arcpy.ListFields(polygon_file)]
    if 'Plant_Id' not in fields:
        arcpy.management.AddField(polygon_file, "Plant_Id", "TEXT", field_length=50)
        print("🛠️ 'Plant_Id' field added to polygon file.")
    
    # Copy Id values to Plant_Id
    with arcpy.da.UpdateCursor(polygon_file, ["Id", "Plant_Id"]) as cursor:
        for row in cursor:
            row[1] = row[0]  # Copy Id to Plant_Id
            cursor.updateRow(row)
    print("✅ 'Plant_Id' values copied from 'Id' in polygon file.")

# Step 2: Update Bed_Row in the Centroid File
def update_bed_row_in_centroid(centroid_file, row_buffer_file):
    # Add the Bed_Row field if it doesn't exist
    fields = [f.name for f in arcpy.ListFields(centroid_file)]
    if 'Bed_Row' not in fields:
        arcpy.management.AddField(centroid_file, "Bed_Row", "LONG")
        print("🛠️ 'Bed_Row' field added to centroid file.")
    
    # Loop through each Bed_Row in the row buffer file
    with arcpy.da.SearchCursor(row_buffer_file, ["SHAPE@", "ORIG_FID"]) as row_cursor:
        for row in row_cursor:
            row_geometry = row[0]
            bed_row_value = row[1]

            print(f"🔄 Processing Bed_Row: {bed_row_value}...")

            # Select centroid points within this Bed_Row
            with arcpy.da.UpdateCursor(centroid_file, ["SHAPE@", "Bed_Row"]) as centroid_cursor:
                for centroid_row in centroid_cursor:
                    centroid_geometry = centroid_row[0]
                    if centroid_geometry.within(row_geometry):
                        centroid_row[1] = bed_row_value  # Update Bed_Row
                        centroid_cursor.updateRow(centroid_row)

            print(f"✅ Centroid file updated for Bed_Row: {bed_row_value}.")

# Step 3: Update Bed_Row in the Polygon File using ORIG_FID from the row buffer
def update_bed_row_in_polygon(polygon_file, row_buffer_file):
    fields = [f.name for f in arcpy.ListFields(polygon_file)]
    if 'Bed_Row' not in fields:
        arcpy.management.AddField(polygon_file, "Bed_Row", "LONG")
        print("🛠️ 'Bed_Row' field added to polygon file.")

    # Update Bed_Row based on the row buffer file
    with arcpy.da.SearchCursor(row_buffer_file, ["SHAPE@", "ORIG_FID"]) as row_cursor:
        for row in row_cursor:
            row_geometry = row[0]
            bed_row_value = row[1]

            print(f"🔄 Updating Bed_Row in polygons for Bed_Row: {bed_row_value}...")

            # Select polygons that are within this Bed_Row
            with arcpy.da.UpdateCursor(polygon_file, ["SHAPE@", "Bed_Row"]) as polygon_cursor:
                for polygon in polygon_cursor:
                    polygon_geometry = polygon[0]
                    if polygon_geometry.within(row_geometry):
                        polygon[1] = bed_row_value  # Update Bed_Row
                        polygon_cursor.updateRow(polygon)

    print("✅ 'Bed_Row' updated in polygon file.")

# Step 4: Create Hexagons and Assign Plant_Id and Bed_Row to Hexagons
def create_hexagons_from_centroids(centroid_file, hexagon_output, hex_size):
    # Create a new feature class to store hexagons
    if not arcpy.Exists(hexagon_output):
        arcpy.CreateFeatureclass_management(os.path.dirname(hexagon_output), os.path.basename(hexagon_output), "POLYGON", spatial_reference=centroid_file)
        print("✅ Hexagon feature class created.")

    # Add Plant_Id and Bed_Row fields if they don't exist
    fields = [f.name for f in arcpy.ListFields(hexagon_output)]
    if 'Plant_Id' not in fields:
        arcpy.management.AddField(hexagon_output, "Plant_Id", "TEXT", field_length=50)
        print("🛠️ 'Plant_Id' field added to hexagon file.")
    if 'Bed_Row' not in fields:
        arcpy.management.AddField(hexagon_output, "Bed_Row", "LONG")
        print("🛠️ 'Bed_Row' field added to hexagon file.")
    
    # Create hexagons and assign fields from centroids
    with arcpy.da.InsertCursor(hexagon_output, ["SHAPE@", "Plant_Id", "Bed_Row"]) as hex_cursor:
        with arcpy.da.SearchCursor(centroid_file, ["SHAPE@XY", "Plant_Id", "Bed_Row"]) as centroid_cursor:
            for centroid in centroid_cursor:
                centroid_xy = centroid[0]
                plant_id_value = centroid[1]
                bed_row_value = centroid[2]

                # Skip centroids that don't have a valid Plant_Id or Bed_Row
                if not plant_id_value or bed_row_value is None:
                    continue

                # Create hexagon around the centroid
                hex_coords = create_hexagon(centroid_xy, hex_size)
                hexagon = arcpy.Polygon(arcpy.Array([arcpy.Point(*coords) for coords in hex_coords]))

                # Insert hexagon with Plant_Id and Bed_Row
                hex_cursor.insertRow([hexagon, plant_id_value, bed_row_value])

    print("✅ Hexagons created and fields assigned based on updated centroid file.")

# Execute all steps
copy_id_to_plant_id(polygon_file)                     # Step 1: Update Plant_Id in polygons
update_bed_row_in_centroid(centroid_file, row_buffer_file)  # Step 2: Update Bed_Row in centroids
update_bed_row_in_polygon(polygon_file, row_buffer_file)    # Step 3: Update Bed_Row in polygons
create_hexagons_from_centroids(centroid_file, hexagon_output, hex_size)  # Step 4: Create hexagons



## Step 7 - Filter plant hexagons to centroid IDs

Keeps only plant hexagons matching centroid Plant_Id values and removes duplicate polygon writes.


In [ ]:
print("🎉 All updates and hexagon creation completed successfully.")

import arcpy
import os

# Define file paths
centroid_shp = os.path.join(
    CENTROID_OUTPUT.as_posix(),
    "Combined_Centroids_Field3_2022_12_07.shp"
)

planthex_shp = os.path.join(
    VECTORS.as_posix(),
    "ERL_Field3_2022_12_07_PlantHex.shp"
)

output_shp = os.path.join(
    VECTORS.as_posix(),
    "Filtered_ERL_Field3_2022_12_07_PlantHex.shp"
)

# Ensure arcpy environment is set correctly
arcpy.env.overwriteOutput = True

# Step 1: Get the Spatial Reference of the PlantHex shapefile
spatial_ref = arcpy.Describe(planthex_shp).spatialReference
if spatial_ref is None or spatial_ref.name == "Unknown":
    print("⚠️ Warning: Projection undefined. Assigning WGS 84 UTM Zone 11N.")
    spatial_ref = arcpy.SpatialReference(32611)  # EPSG: 32611 (WGS84 UTM 11N)

# Step 2: Extract unique Plant_IDs from Centroid Shapefile
centroid_ids = set()
with arcpy.da.SearchCursor(centroid_shp, ["Plant_Id"]) as cursor:
    for row in cursor:
        centroid_ids.add(str(row[0]).strip())  # Convert all to strings

print(f"🟢 Found {len(centroid_ids)} unique Plant_Id values in Centroid file.")

# Step 3: Create a new feature class to store the filtered PlantHex polygons
if arcpy.Exists(output_shp):
    arcpy.Delete_management(output_shp)  # Remove existing output file

arcpy.CreateFeatureclass_management(
    os.path.dirname(output_shp),  # Output folder
    os.path.basename(output_shp),  # File name
    "POLYGON",  # Geometry type
    planthex_shp,  # Copy fields from the original PlantHex file
    "SAME_AS_TEMPLATE",
    "SAME_AS_TEMPLATE",
    spatial_ref  # Apply correct projection
)

# Step 4: Copy attribute fields while ensuring the SHAPE@ geometry is included
fields = [field.name for field in arcpy.ListFields(planthex_shp) if field.type not in ("OID", "Geometry")]
fields.insert(0, "SHAPE@")  # Ensure geometry is copied

# Step 5: Track which PlantHex polygons have already been added (keep only the first occurrence)
kept_count = 0
added_ids = set()  # Prevents duplicate writes

with arcpy.da.SearchCursor(planthex_shp, ["Plant_Id"] + fields) as search_cursor, \
     arcpy.da.InsertCursor(output_shp, ["Plant_Id"] + fields) as insert_cursor:
    for row in search_cursor:
        plant_id = str(row[0]).strip()  # Ensure ID format matches Centroids
        if plant_id in centroid_ids and plant_id not in added_ids:  # Only take first occurrence
            insert_cursor.insertRow(row)  # Ensure geometry is copied
            added_ids.add(plant_id)  # Track inserted Plant_Id
            kept_count += 1  # Track the number of inserted rows

# Step 6: Verify the output file contains features
final_count = int(arcpy.GetCount_management(output_shp)[0])
print(f"✅ Filtering complete! {kept_count} polygons kept in the output file.")
print(f"🎯 Final output shapefile contains {final_count} polygons at: {output_shp}")




## Step 8 - Classify canopy condition from NDRE and redness index

Computes NDRE and a red-green redness index, applies threshold-based healthy/dieback/non-vegetation classification, and saves classification rasters.


In [ ]:

import arcpy
from arcpy.sa import *
import os

# Check out the Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")

# Define workspace and output directory
workspace = PROJECT_ROOT.as_posix()
output_dir = CLASSIFICATION_OUTPUT.as_posix()
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Define classification thresholds based on your observations
dieback_ndre_range = (0.45, 0.59)  
healthy_ndre_range = (0.60, 1.0)
non_veg_ndre_range = (0.14, 0.41)

dieback_ri_range = (0.22, 0.37)  
healthy_ri_range = (-1.0, -0.30)
non_veg_ri_range = (-0.01, 0.16)

# List of multispectral images to be used as inputs. 
multispectral_images = [
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2022_12_07_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_02_09_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_02_20_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_03_28_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_04_21_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_05_15_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_05_22_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_06_01_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_06_10_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_2023_06_20_multispectral.tif")
]

# Loop through each multispectral image
for multispectral_image in multispectral_images:
    # Extract the field number and date from the file name
    filename = os.path.basename(multispectral_image)
    parts = filename.split('_')
    field_number = parts[1]
    date_str = parts[2] + parts[3] + parts[4]  # e.g., 20221207 for 2022_12_07

    print(f"Processing {filename}...")

    # Step 1: Calculate NDRE and Redness Index
    print("🧮 Calculating NDRE and Redness Index...")
    red_band = Raster(multispectral_image + r"/Band_1")  # Assuming Band_1 is Red
    nir_band = Raster(multispectral_image + r"/Band_4")  # Assuming Band_4 is NIR
    green_band = Raster(multispectral_image + r"/Band_2")  # Assuming Band_2 is Green

    # NDRE calculation
    ndre = (nir_band - red_band) / (nir_band + red_band)
    ndre_output_path = os.path.join(output_dir, f"ERL_{field_number}_{date_str}_NDRE.tif")
    ndre.save(ndre_output_path)

    # Redness Index calculation (Normalized Difference Red-Green)
    ri = (red_band - green_band) / (red_band + green_band)
    ri_output_path = os.path.join(output_dir, f"ERL_{field_number}_{date_str}_RI.tif")
    ri.save(ri_output_path)

    # Step 2: Classify based on NDRE and RI
    print("🔍 Classifying based on NDRE and RI...")

    # NDRE classification
    ndre_class = Con((ndre >= healthy_ndre_range[0]) & (ndre <= healthy_ndre_range[1]), 1,  # 1 for Healthy
                     Con((ndre >= dieback_ndre_range[0]) & (ndre <= dieback_ndre_range[1]), 2,  # 2 for Dieback
                     Con((ndre >= non_veg_ndre_range[0]) & (ndre <= non_veg_ndre_range[1]), 3,  # 3 for Non_Veg
                     0)))  # 0 for Unknown

    # RI classification
    ri_class = Con((ri >= healthy_ri_range[0]) & (ri <= healthy_ri_range[1]), 1,  # 1 for Healthy
                   Con((ri >= dieback_ri_range[0]) & (ri <= dieback_ri_range[1]), 2,  # 2 for Dieback
                   Con((ri >= non_veg_ri_range[0]) & (ri <= non_veg_ri_range[1]), 3,  # 3 for Non_Veg
                   0)))  # 0 for Unknown

    # Save classifications separately
    ndre_class_output_path = os.path.join(output_dir, f"ERL_{field_number}_{date_str}_NDRE_Classification.tif")
    ndre_class.save(ndre_class_output_path)

    ri_class_output_path = os.path.join(output_dir, f"ERL_{field_number}_{date_str}_RI_Classification.tif")
    ri_class.save(ri_class_output_path)

    # Step 3: Combine NDRE and RI classifications
    print("🔄 Combining NDRE and RI classifications...")

    final_classification_numeric = Con((ri_class == 0) & (ndre_class == 1), 1,  # 1 for Healthy
                           Con((ri_class == 0) & (ndre_class == 2), 2,  # 2 for Dieback
                           Con((ri_class == 0) & (ndre_class == 3), 3,  # 3 for Non_Veg
                           ri_class)))  # Use RI classification where applicable

    # Step 4: Save the final classification raster
    final_classification_output_path = os.path.join(output_dir, f"ERL_{field_number}_{date_str}_Final_Classification.tif")


## Step 9 - Polygonize classification rasters

Converts final classification rasters to polygons for downstream intersection with plant hexagons.


In [ ]:
    final_classification_numeric.save(final_classification_output_path)
    print(f"✅ Final classification saved to {final_classification_output_path}")

import arcpy
import os

# Check out the Spatial Analyst extension (optional if you have already checked it out)
arcpy.CheckOutExtension("Spatial")

# Define the workspace and output directories
workspace = PROJECT_ROOT.as_posix()
classification_dir = CLASSIFICATION_OUTPUT.as_posix()
polygon_output_dir = POLYGON_OUTPUT.as_posix()
if not os.path.exists(polygon_output_dir):
    os.makedirs(polygon_output_dir)

# List of all multispectral images
classification_rasters = [
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20221207_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230209_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230220_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230328_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230421_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230515_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230522_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230601_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230610_Final_Classification.tif"),
    os.path.join(CLASSIFICATION_OUTPUT.as_posix(), "ERL_Field2_20230620_Final_Classification.tif"),
]

# Loop through each classification raster and polygonize it
for classification_raster in classification_rasters:
    # Extract the field number and date from the file name
    filename = os.path.basename(classification_raster)
    parts = filename.split('_')
    field_number = parts[1]
    date_str = parts[2]  # e.g., 20230610 for 2023_06_10

    print(f"Polygonizing {filename}...")

    # Define the output polygon shapefile path
    polygon_output_path = os.path.join(polygon_output_dir, f"ERL_{field_number}_{date_str}_Final_Classification_Polygon.shp")

    # Polygonize the classification raster
    arcpy.RasterToPolygon_conversion(classification_raster, polygon_output_path, "NO_SIMPLIFY", "VALUE")
    
    print(f"✅ Polygonization completed: {polygon_output_path}")


## Step 10 - Summarize hexagon counts by bed row

Runs a simple bed-row summary table for quality control.


In [ ]:

print("All classifications have been successfully polygonized.")

# First a quick check of the number of hexagons per bed row. 
import arcpy

# Define input and output paths
hexagons = os.path.join(
    VECTORS.as_posix(),
    "ERL_Field2_2022_12_07_PlantHex.shp"
)

output_table = os.path.join(
    VECTORS.as_posix(),
    "ERL_Field2_BedRow_Hexagon_Summary.dbf"
)

# Use a dummy field for the count operation
arcpy.analysis.Statistics(
    hexagons, 
    output_table, 
    statistics_fields=[["Plant_Id", "COUNT"]],  # Use OBJECTID or another numeric field
    case_field="Bed_Row"  # Group by Bed_Row
)

print(f"Summary statistics saved to: {output_table}")


## Step 11 - Extract plant-level canopy metrics, revised in-memory version

Uses ArcPy in_memory layers to reduce layer conflicts while saving final hexagon metrics to disk.


In [ ]:


# Dynamic processing — Assign plant-level canopy metrics using in_memory layers
#
# For each multispectral raster/date:
#   1. Run calculate_canopy_dieback_ndre_ri()
#   2. Clear ArcGIS in_memory workspace before processing
#   3. Process each bed row using temporary in_memory feature layers
#   4. Intersect plant hexagons with canopy classification polygons
#   5. Calculate canopy area metrics and mean NDRE/RI values
#   6. Save the final hexagon metrics shapefile to HEXAGON_TEMP_OUTPUT
#   7. Clear in_memory before moving to the next date
#
# This version is retained as the final canopy metric extraction workflow.

import arcpy
from arcpy.sa import *
import os

# 🧹 Disable auto-adding results to Map during script
arcpy.env.addOutputsToMap = False

# Toggle test mode
test_mode = False
test_row = 26

# Define the paths to the multispectral rasters
multispectral_rasters = [
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20221207_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230209_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230328_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230421_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230515_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230522_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230601_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230610_multispectral.tif"),
    os.path.join(CLIPPED_RASTERS.as_posix(), "ERL_Field2_20230620_multispectral.tif")
]

def calculate_canopy_dieback_ndre_ri(hexagon, classification_polygon, ndre_raster, ri_raster, site="ERL", field="Field3", date="20221207"):
    temp_output_dir = HEXAGON_TEMP_OUTPUT.as_posix()
    if not os.path.exists(temp_output_dir):
        os.makedirs(temp_output_dir)

    # 🧹 Clean up in_memory before starting
    arcpy.management.Delete("in_memory")

    temp_merged_layer = hexagon  # 👉 use input hexagon shapefile directly

    if test_mode:
        bed_rows_to_process = [test_row]
        print(f"🛠 TEST MODE: Processing only Bed_Row {test_row}")
    else:
        bed_rows_to_process = list(range(0, 40))

    for bed_row in bed_rows_to_process:
        print(f"🚧 Processing Bed_Row {bed_row}...")

        hexagon_layer_name = f"in_memory\\hexagon_layer_row_{bed_row}"
        classification_layer_name = f"in_memory\\classification_layer_row_{bed_row}"

        # Make in_memory feature layers
        hexagon_layer = arcpy.management.MakeFeatureLayer(temp_merged_layer, hexagon_layer_name)
        classification_layer = arcpy.management.MakeFeatureLayer(classification_polygon, classification_layer_name)

        # Select by Bed_Row
        arcpy.management.SelectLayerByAttribute(hexagon_layer, "NEW_SELECTION", f"Bed_Row = {bed_row}")

        if int(arcpy.GetCount_management(hexagon_layer)[0]) == 0:
            print(f"⚠️ No features found for Bed_Row {bed_row}. Skipping.")
            arcpy.management.Delete(hexagon_layer)
            arcpy.management.Delete(classification_layer)
            continue

        # 🛠️ Save canopy_intersect to disk (not in_memory!)
        canopy_intersect = os.path.join(temp_output_dir, f"canopy_intersect_Row_{bed_row}.shp")
        if arcpy.Exists(canopy_intersect):
            arcpy.management.Delete(canopy_intersect)

        arcpy.analysis.Intersect([hexagon_layer, classification_layer], canopy_intersect)

        # ✅ AREA METRICS
        if "Shape_Area" not in [f.name for f in arcpy.ListFields(canopy_intersect)]:
            arcpy.management.AddField(canopy_intersect, "Shape_Area", "DOUBLE")
            arcpy.management.CalculateGeometryAttributes(canopy_intersect, [["Shape_Area", "AREA"]])

        with arcpy.da.UpdateCursor(hexagon_layer, ["Plant_Id", "PctHlthy", "PctDieback", "PctNnVeg", "LCnpyArea", "DCnpyArea", "TCnpyArea", "PLCnpA"]) as hex_cursor:
            for row in hex_cursor:
                plant_id = row[0]
                healthy_area = dieback_area = non_veg_area = total_area = 0
                with arcpy.da.SearchCursor(canopy_intersect, ["Plant_Id", "gridcode", "Shape_Area"]) as intersect_cursor:
                    for intersect_row in intersect_cursor:
                        if intersect_row[0] == plant_id:
                            gridcode = intersect_row[1]
                            area = intersect_row[2]
                            if gridcode == 1:
                                healthy_area += area
                            elif gridcode == 2:
                                dieback_area += area
                            elif gridcode == 3:
                                non_veg_area += area
                            total_area += area

                if total_area > 0:
                    row[1] = (healthy_area / total_area) * 100
                    row[2] = (dieback_area / total_area) * 100
                    row[3] = (non_veg_area / total_area) * 100
                    row[4] = (row[1] / 100) * 0.075
                    row[5] = (row[2] / 100) * 0.075
                    row[6] = row[4] + row[5]
                    row[7] = (row[4] / row[6]) * 100 if row[6] > 0 else 0
                    hex_cursor.updateRow(row)

        print(f"✅ Area metrics updated for Bed_Row {bed_row}.")

        # ✅ NDRE & RI STATS
        if arcpy.Exists(ndre_raster) and arcpy.Exists(ri_raster):
            print(f"📊 Extracting NDRE & RI values for Bed_Row {bed_row}...")
            canopy_masked_ndre = ExtractByMask(ndre_raster, canopy_intersect)
            canopy_masked_ri = ExtractByMask(ri_raster, canopy_intersect)

            if canopy_masked_ndre is not None and canopy_masked_ri is not None:
                canopy_ndre_raster = f"in_memory\\canopy_NDRE_Row_{bed_row}"
                canopy_ri_raster = f"in_memory\\canopy_RI_Row_{bed_row}"
                canopy_masked_ndre.save(canopy_ndre_raster)
                canopy_masked_ri.save(canopy_ri_raster)

                # Zonal stats NDRE
                ndre_table = f"in_memory\\zonal_stats_NDRE_Row_{bed_row}"
                ZonalStatisticsAsTable(hexagon_layer, "Plant_Id", canopy_ndre_raster, ndre_table, "DATA", "MEAN")

                if arcpy.Exists(ndre_table) and int(arcpy.GetCount_management(ndre_table)[0]) > 0:
                    arcpy.management.JoinField(hexagon_layer, "Plant_Id", ndre_table, "Plant_Id", ["MEAN"])
                    arcpy.management.CalculateField(hexagon_layer, "MeanNDRE", "!MEAN!", "PYTHON3")
                    arcpy.management.DeleteField(hexagon_layer, ["MEAN"])
                else:
                    print(f"⚠️ Warning: No valid NDRE zonal stats for Bed_Row {bed_row}. Skipping NDRE join.")

                # Zonal stats RI
                ri_table = f"in_memory\\zonal_stats_RI_Row_{bed_row}"
                ZonalStatisticsAsTable(hexagon_layer, "Plant_Id", canopy_ri_raster, ri_table, "DATA", "MEAN")

                if arcpy.Exists(ri_table) and int(arcpy.GetCount_management(ri_table)[0]) > 0:
                    arcpy.management.JoinField(hexagon_layer, "Plant_Id", ri_table, "Plant_Id", ["MEAN"])
                    arcpy.management.CalculateField(hexagon_layer, "MeanRI", "!MEAN!", "PYTHON3")
                    arcpy.management.DeleteField(hexagon_layer, ["MEAN"])
                else:
                    print(f"⚠️ Warning: No valid RI zonal stats for Bed_Row {bed_row}. Skipping RI join.")

                print(f"✅ Mean NDRE and RI updated for Bed_Row {bed_row}.")

        # ✅ Centroids
        arcpy.management.CalculateGeometryAttributes(
            hexagon_layer,
            [["Easting", "CENTROID_X"], ["Northing", "CENTROID_Y"]],
            coordinate_system=arcpy.Describe(hexagon).spatialReference
        )
        print(f"✅ Computed Easting & Northing for Bed_Row {bed_row}.")

        # 🧹 Clean up memory layers after each Bed_Row
        arcpy.management.Delete(hexagon_layer)
        arcpy.management.Delete(classification_layer)
        arcpy.management.Delete(canopy_intersect)

    # ✅ Save updated hexagons
    output_hexagon = os.path.join(temp_output_dir, f"{site}_{field}_{date}_HexMetrics_TEST.shp" if test_mode else f"{site}_{field}_{date}_HexMetrics.shp")
    arcpy.management.CopyFeatures(temp_merged_layer, output_hexagon)
    print(f"💾 Saved updated hexagon layer to {output_hexagon}.")

    # Clean up in_memory
    arcpy.management.Delete("in_memory")

# Main run loop
for multispectral in multispectral_rasters:
    filename = os.path.basename(multispectral)
    parts = filename.split('_')
    field = parts[1]
    date = parts[2]
    hexagon = os.path.join(VECTORS.as_posix(), f"ERL_{field}_2022_12_07_PlantHex.shp")
classification_polygon = os.path.join(POLYGON_OUTPUT.as_posix(), f"ERL_{field}_{date}_Final_Classification_Polygon.shp")
ndre_raster = os.path.join(CLASSIFICATION_OUTPUT.as_posix(), f"ERL_{field}_{date}_NDRE.tif")
ri_raster = os.path.join(CLASSIFICATION_OUTPUT.as_posix(), f"ERL_{field}_{date}_RI.tif")

    if all(os.path.exists(p) for p in [hexagon, classification_polygon, ndre_raster, ri_raster]):
        print(f"\n📦 Processing {field} for {date}...")
        calculate_canopy_dieback_ndre_ri(hexagon, classification_polygon, ndre_raster, ri_raster, site="ERL", field=field, date=date)
    else:
        print(f"⚠️ Missing files for {field} on {date}, skipping.")

print("\n✅ Finished processing all fields and dates.")

# ✅ Re-enable auto-adding outputs to Map after script finishes
arcpy.env.addOutputsToMap = True






## Output

The final output of the ArcGIS Pro preprocessing workflow is a collection of
plant-level spatial datasets containing canopy geometry, vegetation indices,
and canopy health metrics.

These datasets serve as the input for the Python data integration workflow
described in the next stage of the repository.

→ `scripts/02_python_processing`